In [2]:
import pandas as pd

cancer_variant_matrix = pd.read_csv(f"../file_merging/Updated/cancer_variant_matrix_normalized.csv", index_col=0)
#top_20_variants_cancers = cancer_variant_matrix.stack().sort_values(ascending=False).head(20)

top_20_variants_cancers = (
    cancer_variant_matrix
    .stack()
    .rename("score")
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={
        "level_0": "cancer",
        "level_1": "variant"
    })
)

top_20_variants_cancers["score"] = top_20_variants_cancers["score"].round(3)
top_20_variants_cancers.insert(0, "rank", range(1, 21))
#top_20_variants_cancers.to_csv('../../score_evaluation/top_20_variants_cancers.csv', index=False)
top_20_variants_cancers

,rank,cancer,variant,score
0,1,neurofibroma,v600e_BRAF,100.000
1,2,myofibroma,r561c_PDGFRB,100.000
2,3,myofibroma,d850v_PDGFRB,100.000
3,4,histiocytoma,d459g_BRAF,100.000
4,5,histiocytoma,v600e_BRAF,100.000
5,6,ocular melanoma,v600e_BRAF,100.000
6,7,myofibroma,n666k_PDGFRB,100.000
7,8,synovial sarcoma,r248q_TP53,100.000
8,9,clear cell sarcoma,v600e_BRAF,100.000
9,10,chordoma,t910a_PARP1,100.000


In [12]:
import networkx as nx
import pandas as pd
import numpy as np
import json

G = nx.read_gml('../file_merging/Updated/network_graph_weighted.gml')

######## UPDATED

#### Updated weighted network graph with automated threshold, based on qualitative analysis

variant_of_interest = "v600e_BRAF"
cancer_of_interest = "ganglioglioma"

# Adjustable thresholds
TREATMENT_THRESHOLD_PERCENTILE = 80    # highlight top X% of treatment weights
TREATMENT_MIN_HIGHLIGHT        = 300   # and require ≥X total weight
CANCER_THRESHOLD_PERCENTILE    = 80    # highlight top X% of cancer–variant weights
CANCER_MIN_HIGHLIGHT           = 80    # and require ≥X total weight

df_consensus = pd.read_csv("../file_merging/Updated/final_variant_treatment_consensus.csv")

# Prepare consensus lookup
df_consensus["Variant_Treatment_Pair"] = (
    df_consensus["Variant_Treatment_Pair"]
    .str.strip()
    .str.lower()
)
consensus_dict = dict(
    zip(df_consensus["Variant_Treatment_Pair"], df_consensus["Resolved_Prediction"])
)

excluded_treatments = {
    'chemotherapy', 'tyrosine kinase inhibitor', 'radiotherapy', 'hormone therapy',
    'adjuvant chemotherapy', 'immunotherapy', 'immune checkpoint inhibitor', 'adjuvant chemotherapy',
    'mrna vaccine', 'mtor inhibitor', 'radiation ionizing radiotherapy', 
    'braf inhibitor','angiogenesis inhibitor', 'aromatase inhibitor', 'bet inhibitor',
    'EGFR tyrosine kinase inhibitor therapy', 'epidermal growth factor receptor tyrosine kinase inhibitor',
    'hematopoietic cell transplantation', 'hyperthermic intraperitoneal chemotherapy', 'TRK inhibitor',
    'tyrosine kinase inhibitor', 'therapeutic tumor infiltrating lymphocytes'
}

# Cancer‐only treatments
canc_nei = set(G.neighbors(cancer_of_interest))
treatments = [
    n for n in canc_nei
    if G.nodes[n]['category']=='Treatment'
    and n.lower() not in excluded_treatments
]
t_weights = {t: G[cancer_of_interest][t]['weight'] for t in treatments}
top_cancer_treats = sorted(t_weights.items(), key=lambda x: x[1], reverse=True)[:6]
c_w = list(t_weights.values())
treat_pct = np.percentile(c_w, TREATMENT_THRESHOLD_PERCENTILE) if c_w else 0


# Variant + cancer associations
sensitive, resistant = [], []
for t in treatments:
    try:
        w = G[cancer_of_interest][t]['weight'] + G[variant_of_interest][t]['weight']
        pred = consensus_dict.get(f"{variant_of_interest} + {t}".lower())
        if pred == "Sensitive":
            sensitive.append((t, w))
        elif pred == "Resistant":
            resistant.append((t, w))
    except KeyError:
        continue

top_sens = sorted(sensitive, key=lambda x: x[1], reverse=True)[:6]
top_res  = sorted(resistant, key=lambda x: x[1], reverse=True)[:6]
sens_w = [w for _, w in sensitive]
res_w  = [w for _, w in resistant]
sens_pct = np.percentile(sens_w, TREATMENT_THRESHOLD_PERCENTILE) if sens_w else 0
res_pct  = np.percentile(res_w,   TREATMENT_THRESHOLD_PERCENTILE) if res_w else 0

sens_strong_json = []
sens_weak_json = []
res_strong_json = []
res_weak_json = []

print(f"\n\033[1mSensitive treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_sens:
    if w >= sens_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        sens_strong_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[1;32m{t}: {w:.0f}\033[0m")
    else:
        sens_weak_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

print(f"\n\033[1mResistant treatments for variant '{variant_of_interest}' "
      f"(≥{TREATMENT_THRESHOLD_PERCENTILE}th pct & ≥{TREATMENT_MIN_HIGHLIGHT}):\033[0m")
for t, w in top_res:
    if w >= res_pct and w >= TREATMENT_MIN_HIGHLIGHT:
        res_strong_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[1;31m{t}: {w:.0f}\033[0m")
    else:
        res_weak_json.append({"treatment": t, "weight": round(w)})
        print(f"\033[2;37m{t}: {w:.0f}\033[0m")

df = pd.DataFrame({
    "variant": variant_of_interest,
    "cancer": cancer_of_interest,
    "sensitive_treatments_strong": json.dumps(sens_strong_json),
    "sensitive_treatments_weak": json.dumps(sens_weak_json),
    "resistant_treatments_strong": json.dumps(res_strong_json),
    "resistant_treatments_weak": json.dumps(res_weak_json),
}, index=[0])
df


Sensitive treatments for variant 'v600e_BRAF' (≥80th pct & ≥300):
Vemurafenib: 453
Dabrafenib: 376
Trametinib: 342
Dabrafenib/Trametinib Regimen: 179
Cobimetinib: 64
Temozolomide: 47

Resistant treatments for variant 'v600e_BRAF' (≥80th pct & ≥300):
Radiation Therapy: 222
Carboplatin: 30
Vincristine: 16
Larotrectinib: 14


,variant,cancer,sensitive_treatments_strong,sensitive_treatments_weak,resistant_treatments_strong,resistant_treatments_weak
0,v600e_BRAF,ganglioglioma,"[{""treatment"": ""Vemurafenib"", ""weight"": 453}, ...","[{""treatment"": ""Trametinib"", ""weight"": 342}, {...",[],"[{""treatment"": ""Radiation Therapy"", ""weight"": ..."


## Take top 20 variant-cancer pairs for which there is a treatment

In [ ]:
import pandas as pd
import networkx as nx
import pandas as pd
import numpy as np
import json

def get_treatments(G, variant_of_interest, cancer_of_interest, score, excluded_treatments, consensus_dict):
    # Cancer‐only treatments
    canc_nei = set(G.neighbors(cancer_of_interest))
    treatments = [
        n for n in canc_nei
        if G.nodes[n]['category']=='Treatment'
        and n.lower() not in excluded_treatments
    ]
    t_weights = {t: G[cancer_of_interest][t]['weight'] for t in treatments}
    top_cancer_treats = sorted(t_weights.items(), key=lambda x: x[1], reverse=True)[:6]
    c_w = list(t_weights.values())
    treat_pct = np.percentile(c_w, TREATMENT_THRESHOLD_PERCENTILE) if c_w else 0


    # Variant + cancer associations
    sensitive, resistant = [], []
    for t in treatments:
        try:
            w = G[cancer_of_interest][t]['weight'] + G[variant_of_interest][t]['weight']
            pred = consensus_dict.get(f"{variant_of_interest} + {t}".lower())
            if pred == "Sensitive":
                sensitive.append((t, w))
            elif pred == "Resistant":
                resistant.append((t, w))
        except KeyError:
            continue

    top_sens = sorted(sensitive, key=lambda x: x[1], reverse=True)[:6]
    top_res  = sorted(resistant, key=lambda x: x[1], reverse=True)[:6]
    sens_w = [w for _, w in sensitive]
    res_w  = [w for _, w in resistant]
    sens_pct = np.percentile(sens_w, TREATMENT_THRESHOLD_PERCENTILE) if sens_w else 0
    res_pct  = np.percentile(res_w,   TREATMENT_THRESHOLD_PERCENTILE) if res_w else 0

    sens_strong_json = []
    sens_weak_json = []
    res_strong_json = []
    res_weak_json = []


    for t, w in top_sens:
        if w >= sens_pct and w >= TREATMENT_MIN_HIGHLIGHT:
            sens_strong_json.append({"treatment": t, "weight": round(w)})
        else:
            sens_weak_json.append({"treatment": t, "weight": round(w)})

    for t, w in top_res:
        if w >= res_pct and w >= TREATMENT_MIN_HIGHLIGHT:
            res_strong_json.append({"treatment": t, "weight": round(w)})
        else:
            res_weak_json.append({"treatment": t, "weight": round(w)})

    #if len(sens_strong_json) + len(sens_weak_json) + len(res_strong_json) + len(res_weak_json) > 0:
    if len(sens_strong_json) + len(res_strong_json) > 0:
        df = pd.DataFrame({
            "variant": variant_of_interest,
            "cancer": cancer_of_interest,
            "score": score,
            "sensitive_treatments_strong": json.dumps(sens_strong_json),
            "sensitive_treatments_weak": json.dumps(sens_weak_json),
            "resistant_treatments_strong": json.dumps(res_strong_json),
            "resistant_treatments_weak": json.dumps(res_weak_json),
        }, index=[0])
    else:
        df = None
    return df

# Adjustable thresholds
TREATMENT_THRESHOLD_PERCENTILE = 80    # highlight top X% of treatment weights
TREATMENT_MIN_HIGHLIGHT        = 300   # and require ≥X total weight
CANCER_THRESHOLD_PERCENTILE    = 80    # highlight top X% of cancer–variant weights
CANCER_MIN_HIGHLIGHT           = 80    # and require ≥X total weight

df_consensus = pd.read_csv("../file_merging/Updated/final_variant_treatment_consensus.csv")
cancer_variant_matrix = pd.read_csv(f"../file_merging/Updated/cancer_variant_matrix_normalized.csv", index_col=0)

G = nx.read_gml('../file_merging/Updated/network_graph_weighted.gml')

top_variants_cancers = (
    cancer_variant_matrix
    .stack()
    .rename("score")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={
        "level_0": "cancer",
        "level_1": "variant"
    })
)

top_variants_cancers["score"] = top_variants_cancers["score"].round(3)

# Prepare consensus lookup
df_consensus["Variant_Treatment_Pair"] = (
    df_consensus["Variant_Treatment_Pair"]
    .str.strip()
    .str.lower()
)
consensus_dict = dict(
    zip(df_consensus["Variant_Treatment_Pair"], df_consensus["Resolved_Prediction"])
)

excluded_treatments = {
    'chemotherapy', 'tyrosine kinase inhibitor', 'radiotherapy', 'hormone therapy',
    'adjuvant chemotherapy', 'immunotherapy', 'immune checkpoint inhibitor', 'adjuvant chemotherapy',
    'mrna vaccine', 'mtor inhibitor', 'radiation ionizing radiotherapy', 
    'braf inhibitor','angiogenesis inhibitor', 'aromatase inhibitor', 'bet inhibitor',
    'EGFR tyrosine kinase inhibitor therapy', 'epidermal growth factor receptor tyrosine kinase inhibitor',
    'hematopoietic cell transplantation', 'hyperthermic intraperitoneal chemotherapy', 'TRK inhibitor',
    'tyrosine kinase inhibitor', 'therapeutic tumor infiltrating lymphocytes'
}

dfs = []
for _, row in top_variants_cancers.iterrows():
    df = get_treatments(
        G,
        row['variant'],
        row['cancer'],
        row['score'],
        excluded_treatments,
        consensus_dict
    )

    if df is not None:
        dfs.append(df)
        if len(dfs) == 20:
            break

result_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
#result_df.to_csv('./top_20_variants_cancers.csv', index=False)
result_df

,variant,cancer,score,sensitive_treatments_strong,sensitive_treatments_weak,resistant_treatments_strong,resistant_treatments_weak
0,v600e_BRAF,neurofibroma,100.000,"[{""treatment"": ""Trametinib"", ""weight"": 338}]",[],[],[]
1,v600e_BRAF,histiocytoma,100.000,"[{""treatment"": ""Vemurafenib"", ""weight"": 444}]",[],[],"[{""treatment"": ""Cisplatin"", ""weight"": 34}, {""t..."
2,v600e_BRAF,ocular melanoma,100.000,"[{""treatment"": ""Vemurafenib"", ""weight"": 444}]",[],[],[]
3,v600e_BRAF,clear cell sarcoma,100.000,"[{""treatment"": ""Vemurafenib"", ""weight"": 444}]","[{""treatment"": ""Cetuximab"", ""weight"": 146}]",[],[]
4,v600e_BRAF,spitzoid melanoma,100.000,"[{""treatment"": ""Trametinib"", ""weight"": 338}]","[{""treatment"": ""Sorafenib Tosylate"", ""weight"":...",[],[]
5,v600e_BRAF,ganglioglioma,93.593,"[{""treatment"": ""Vemurafenib"", ""weight"": 453}, ...","[{""treatment"": ""Trametinib"", ""weight"": 342}, {...",[],"[{""treatment"": ""Radiation Therapy"", ""weight"": ..."
6,v600e_BRAF,pleomorphic xanthoastrocytoma,91.864,"[{""treatment"": ""Vemurafenib"", ""weight"": 446}, ...","[{""treatment"": ""Dabrafenib/Trametinib Regimen""...",[],"[{""treatment"": ""Radiation Therapy"", ""weight"": ..."
7,v600e_BRAF,melanoma,73.683,"[{""treatment"": ""Vemurafenib"", ""weight"": 731}, ...","[{""treatment"": ""Dabrafenib/Trametinib Regimen""...",[],"[{""treatment"": ""Radiation Therapy"", ""weight"": ..."
8,v600e_BRAF,transitional cell cancer,72.759,"[{""treatment"": ""Vemurafenib"", ""weight"": 446}]","[{""treatment"": ""Sorafenib Tosylate"", ""weight"":...",[],[]
9,v600e_BRAF,astrocytoma,70.108,"[{""treatment"": ""Vemurafenib"", ""weight"": 451}, ...","[{""treatment"": ""Dabrafenib/Trametinib Regimen""...",[],"[{""treatment"": ""Radiation Therapy"", ""weight"": ..."
